# 02 - Silver/Gold layer + AI
Builds the analytics behind the Command Center app, dashboard and Genie space.
* **silver_tickets** - issue domain + issue signature extraction (error codes, AUTH/BILL codes, show/episode/subtitle language) across EN/ES/PT/DE/FR text
* **silver_country_streams_5m / gold_release_uplift_5m** - release-calendar-aware stream baseline (simulcasts are not anomalies)
* **gold_service_health_country_5m / _region_5m** - health matrix: Healthy / Watch / Customer-Detected / Telemetry Alert / Known Incident / Recovering
* **gold_social_posts** - `ai_analyze_sentiment` + `ai_classify` on Reddit/X posts
* **gold_payment_health_5m, gold_region_heatmap_5m, gold_frustration_index_5m**
* **gold_signature_trends_15m, silver_anomaly_events, gold_ticket_anomalies** - surge detection + `ai_query` incident summaries, root cause, actions, customer macro
* **gold_incident_scorecard, gold_exec_daily_kpis** - detection lead time and executive KPIs

In [0]:
FQ="classic_stable_l0m1dn_catalog.support_command_center"
spark.sql(f"USE {FQ}")
spark.sql(r"""
CREATE OR REPLACE TABLE silver_tickets AS
WITH b AS (
 SELECT t.*, r.region_group, CAST(from_unixtime(floor(unix_timestamp(created_at)/300)*300) AS TIMESTAMP) AS ts_5m,
   lower(concat(subject,' ',description)) AS txt
 FROM support_tickets t JOIN regions r USING (country_code)),
c AS (
 SELECT *,
  CASE WHEN txt rlike 'subt[ií]t|sous-titres|subtitle|subs |dub|audio' THEN 'Subtitles & Localization'
       WHEN txt rlike 'payment|charged|card|bill|invoice|zahlung|renew|google play|cancel|price' THEN 'Billing & Payments'
       WHEN txt rlike 'log|sign|password|auth-' THEN 'Authentication'
       WHEN txt rlike 'search' THEN 'Search & Discovery'
       WHEN txt rlike 'crash|freez' THEN 'Device Apps'
       WHEN txt rlike 'not available|new episode|episode not showing' THEN 'Content Ingestion'
       ELSE 'Playback & CDN' END AS issue_domain,
  nullif(regexp_extract(txt,'(?:error|erro|erreur)\\s*(?:code\\s*)?(\\d{4})',1),'') AS error_code,
  nullif(upper(regexp_extract(txt,'((?:auth|bill)-\\d{3})',1)),'') AS service_code,
  nullif(regexp_extract(concat(subject,' ',description),'(Blade of the Crimson Moon|Spirit Ronin|Iron Samurai Gaiden|Moonlit Idol Stage|Mecha Academy Zero|Isekai Cafe Chronicles|Shadow Guild Hunters|Ocean Spirit Diaries)',1),'') AS show_title,
  nullif(regexp_extract(txt,'(?:episode|episodio|épisode|ep)\\s*(\\d+)',1),'') AS episode_number,
  CASE WHEN txt rlike 'spanish|español|espanol' THEN 'Spanish' WHEN txt rlike 'sous-titres|french' THEN 'French' END AS subtitle_language
 FROM b)
SELECT ticket_id, created_at, ts_5m, country_code, region_group, language, device, channel, subscriber_tier, form_category,
  subject, description, issue_domain, error_code, service_code, show_title, CAST(episode_number AS INT) episode_number, subtitle_language,
  CASE WHEN error_code IS NOT NULL THEN concat('Error ', error_code)
   WHEN service_code IS NOT NULL THEN concat(service_code, ' · ', issue_domain)
   WHEN issue_domain='Subtitles & Localization' AND subtitle_language IS NOT NULL AND show_title IS NOT NULL
        THEN concat(CASE WHEN txt rlike 'sync|décal|timing|late|behind' THEN 'Subtitles out of sync · ' ELSE 'Subtitles missing · ' END, show_title,' E',episode_number,' · ',subtitle_language)
   WHEN issue_domain='Billing & Payments' AND txt rlike 'declined|failed|fehlgeschlagen|abgelehnt|could not be processed' THEN concat('Payment failure · ', country_code)
   ELSE concat(issue_domain,' · general') END AS issue_signature,
  first_response_minutes, sla_target_minutes, first_response_minutes <= sla_target_minutes AS sla_met, resolved_at, csat_score, reopened
FROM c""")
spark.sql("""
CREATE OR REPLACE TABLE gold_release_uplift_5m AS
WITH t AS (SELECT DISTINCT ts FROM concurrent_streams),
j AS (SELECT t.ts, r.show_title, r.episode_number, r.release_type, r.expected_uplift, (unix_timestamp(t.ts)-unix_timestamp(r.release_ts))/60.0 AS m
      FROM t LEFT JOIN release_schedule r ON t.ts >= r.release_ts AND t.ts <= r.release_ts + INTERVAL 360 MINUTES)
SELECT ts, coalesce(sum(expected_uplift*least(1,(m+5)/30)*exp(-greatest(0,m-30)/80)),0) AS expected_release_uplift,
       max_by(concat(show_title,' E',episode_number, CASE WHEN release_type='Season Finale' THEN ' (Finale)' ELSE '' END), expected_uplift) AS active_release
FROM j GROUP BY ts""")
spark.sql("""
CREATE OR REPLACE TABLE silver_country_streams_5m AS
WITH s AS (SELECT ts, country_code, sum(concurrent_streams) streams, sum(playback_start_failures) playback_start_failures FROM concurrent_streams GROUP BY ALL),
d AS (SELECT s.*, u.expected_release_uplift, u.active_release, s.streams/(1+u.expected_release_uplift) AS deseason, hour(s.ts)*60+minute(s.ts) AS slot
      FROM s JOIN gold_release_uplift_5m u USING (ts))
SELECT ts, country_code, streams, playback_start_failures, expected_release_uplift, active_release,
  round((sum(deseason) OVER (PARTITION BY country_code, slot) - deseason) / (count(*) OVER (PARTITION BY country_code, slot) - 1) * (1+expected_release_uplift)) AS baseline_streams
FROM d""")

In [0]:
def health_sql(grain, tbl):
    key = "country_code" if grain=="country" else "region_group"
    inc_match = "array_contains(split(i.countries,','), h.scope)" if grain=="country" else "array_contains(split(i.region_groups,','), h.scope)"
    return f"""
CREATE OR REPLACE TABLE {tbl} AS
WITH sm AS (SELECT m.ts, r.{key} AS scope, m.domain, sum(m.request_count) req, sum(m.error_rate*m.request_count)/sum(m.request_count) error_rate, avg(m.p95_latency_ms) p95_latency_ms
  FROM service_metrics m JOIN regions r USING (country_code) GROUP BY ALL),
st AS (SELECT s.ts, r.{key} AS scope, sum(streams) streams FROM silver_country_streams_5m s JOIN regions r USING (country_code) GROUP BY ALL),
tk AS (SELECT ts_5m ts, {key} AS scope, issue_domain domain, count(*) tickets FROM silver_tickets GROUP BY ALL),
j AS (SELECT sm.*, st.streams, coalesce(tk.tickets,0) tickets FROM sm JOIN st USING (ts, scope) LEFT JOIN tk USING (ts, scope, domain)),
daily AS (SELECT scope, domain, date(ts) d, sum(tickets)/sum(streams) cr FROM j GROUP BY ALL),
bl AS (SELECT scope, domain, percentile_approx(cr,0.5) contact_rate_baseline FROM daily GROUP BY ALL),
eb AS (SELECT scope, domain, percentile_approx(error_rate,0.5) error_rate_baseline FROM j GROUP BY ALL),
r AS (SELECT j.*, bl.contact_rate_baseline, eb.error_rate_baseline, sum(tickets) OVER w tickets_15m, sum(streams) OVER w streams_15m, avg(error_rate) OVER w error_rate_15m
      FROM j JOIN bl USING (scope, domain) JOIN eb USING (scope, domain)
      WINDOW w AS (PARTITION BY scope, domain ORDER BY ts ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)),
m AS (SELECT *, streams_15m*contact_rate_baseline AS expected_tickets_15m, (tickets_15m+1)/(streams_15m*contact_rate_baseline+1) AS ticket_ratio,
        (tickets_15m - streams_15m*contact_rate_baseline)/sqrt(streams_15m*contact_rate_baseline+1) AS ticket_z, error_rate_15m/error_rate_baseline AS error_ratio FROM r),
h AS (SELECT m.*, (tickets_15m>=8 AND ticket_ratio>=2.5 AND ticket_z>=4) AS customer_signal, (tickets_15m>=20 AND ticket_ratio>=5) AS customer_signal_severe,
        (error_ratio>=3) AS telemetry_signal, (error_ratio>=8) AS telemetry_signal_severe FROM m),
hi AS (SELECT h.*,
        max_by(i.incident_id, i.declared_at) FILTER (WHERE h.ts >= i.declared_at AND h.ts < i.resolved_at) AS open_incident_id,
        max_by(i.incident_id, i.declared_at) FILTER (WHERE h.ts >= i.resolved_at AND h.ts < i.resolved_at + INTERVAL 30 MINUTES) AS recovering_incident_id
      FROM h LEFT JOIN incidents i ON i.domain=h.domain AND {inc_match}
        AND h.ts >= i.declared_at - INTERVAL 1 MINUTE AND h.ts < i.resolved_at + INTERVAL 30 MINUTES
      GROUP BY ALL)
SELECT ts, scope AS {key}, domain, streams, req AS request_count, round(error_rate,5) error_rate, round(error_rate_baseline,5) error_rate_baseline,
  round(error_ratio,2) error_ratio, round(p95_latency_ms,1) p95_latency_ms, tickets, tickets_15m, round(expected_tickets_15m,2) expected_tickets_15m,
  round(ticket_ratio,2) ticket_ratio, round(ticket_z,2) ticket_z, customer_signal, telemetry_signal, open_incident_id, recovering_incident_id,
  CASE WHEN open_incident_id IS NOT NULL THEN 'Known Incident'
       WHEN recovering_incident_id IS NOT NULL AND (customer_signal OR telemetry_signal) THEN 'Recovering'
       WHEN customer_signal AND NOT telemetry_signal THEN 'Customer-Detected'
       WHEN telemetry_signal THEN 'Telemetry Alert'
       WHEN tickets_15m>=5 AND ticket_ratio>=1.8 THEN 'Watch' ELSE 'Healthy' END AS health_state,
  CASE WHEN customer_signal_severe OR telemetry_signal_severe THEN 'Outage' WHEN customer_signal OR telemetry_signal THEN 'Degraded' ELSE 'Normal' END AS severity,
  greatest(0, round(100 - least(60, 12*greatest(0,ticket_ratio-1)) - least(40, 6*greatest(0,error_ratio-1)))) AS health_score
FROM hi"""
spark.sql(health_sql("country","gold_service_health_country_5m"))
spark.sql(health_sql("region","gold_service_health_region_5m"))

In [0]:
%sql
CREATE OR REPLACE TABLE classic_stable_l0m1dn_catalog.support_command_center.gold_social_posts AS
SELECT p.*, r.region_group,
  ai_analyze_sentiment(post_text) AS sentiment,
  ai_classify(post_text, ARRAY('Authentication','Billing & Payments','Playback & CDN','Content Ingestion','Subtitles & Localization','Device Apps','Search & Discovery','Show hype / fandom','General feedback')) AS topic_domain,
  CAST(from_unixtime(floor(unix_timestamp(posted_at)/900)*900) AS TIMESTAMP) AS ts_15m
FROM classic_stable_l0m1dn_catalog.support_command_center.social_posts p
LEFT JOIN classic_stable_l0m1dn_catalog.support_command_center.regions r USING (country_code)

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE gold_payment_health_5m AS
WITH p AS (SELECT p.*, r.region_group FROM payment_transactions p JOIN regions r USING (country_code)),
b AS (SELECT country_code, gateway, sum(successes)/sum(attempts) baseline_success_rate FROM p WHERE date(ts) < '2026-10-24' GROUP BY ALL),
w AS (SELECT p.*, b.baseline_success_rate, sum(attempts) OVER x att_15m, sum(successes) OVER x succ_15m, sum(gateway_errors) OVER x gw_err_15m, sum(customer_declines) OVER x cust_dec_15m
 FROM p JOIN b USING (country_code, gateway) WINDOW x AS (PARTITION BY country_code, gateway ORDER BY ts ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)),
t AS (SELECT ts_5m ts, country_code, count(*) billing_tickets FROM silver_tickets WHERE issue_domain='Billing & Payments' GROUP BY ALL),
s AS (SELECT w.*, CASE WHEN gw_err_15m>=4 AND gw_err_15m/att_15m>=0.10 THEN 'Critical' WHEN gw_err_15m>=3 AND gw_err_15m/att_15m>=0.03 THEN 'Degraded' ELSE 'Healthy' END gateway_state FROM w)
SELECT s.ts, s.country_code, s.region_group, s.gateway, attempts, successes, customer_declines, gateway_errors,
  round(successes/nullif(attempts,0),4) success_rate, round(succ_15m/nullif(att_15m,0),4) success_rate_15m, round(baseline_success_rate,4) baseline_success_rate,
  round(gw_err_15m/nullif(att_15m,0),4) gateway_error_rate_15m, gw_err_15m, cust_dec_15m,
  CASE WHEN gateway_state<>'Healthy' THEN round(gateway_errors*avg_price_usd,2) ELSE 0 END revenue_at_risk_usd,
  coalesce(t.billing_tickets,0) country_billing_tickets, gateway_state
FROM s LEFT JOIN t USING (ts, country_code)""")
spark.sql("""
CREATE OR REPLACE TABLE gold_region_heatmap_5m AS
WITH h AS (SELECT ts, country_code, sum(tickets) tickets, sum(tickets_15m) tickets_15m, sum(expected_tickets_15m) expected_tickets_15m, max_by(domain, ticket_ratio) top_domain,
             max(CASE health_state WHEN 'Customer-Detected' THEN 5 WHEN 'Known Incident' THEN 4 WHEN 'Telemetry Alert' THEN 3 WHEN 'Recovering' THEN 2 WHEN 'Watch' THEN 1 ELSE 0 END) worst_state_rank,
             max(CASE WHEN domain='Authentication' THEN error_rate END) login_failure_rate, max(CASE WHEN domain='Playback & CDN' THEN error_rate END) playback_error_rate
           FROM gold_service_health_country_5m GROUP BY ALL),
p AS (SELECT ts, country_code, sum(successes)/nullif(sum(attempts),0) payment_success_rate, sum(revenue_at_risk_usd) revenue_at_risk_usd FROM gold_payment_health_5m GROUP BY ALL)
SELECT s.ts, s.country_code, r.country_name, r.region_group, r.lat, r.lon, r.subscribers, s.streams, s.baseline_streams, round(s.streams/s.baseline_streams-1,4) stream_deviation_pct, s.active_release,
  h.tickets, h.tickets_15m, round(h.expected_tickets_15m,1) expected_tickets_15m, round(h.tickets_15m/(r.subscribers/1e5),2) tickets_15m_per_100k_subs,
  round((h.tickets_15m+1)/(h.expected_tickets_15m+1),2) ticket_deviation_ratio, round((h.tickets_15m-h.expected_tickets_15m)/sqrt(h.expected_tickets_15m+1),2) ticket_z,
  h.top_domain, element_at(array('Healthy','Watch','Recovering','Telemetry Alert','Known Incident','Customer-Detected'), h.worst_state_rank+1) worst_health_state,
  round(h.login_failure_rate,5) login_failure_rate, round(h.playback_error_rate,5) playback_error_rate, round(p.payment_success_rate,4) payment_success_rate, round(p.revenue_at_risk_usd,2) revenue_at_risk_usd,
  CASE WHEN (h.tickets_15m+1)/(h.expected_tickets_15m+1)>=4 AND h.tickets_15m>=15 THEN 'Critical' WHEN (h.tickets_15m+1)/(h.expected_tickets_15m+1)>=2 AND h.tickets_15m>=8 THEN 'Elevated' ELSE 'Normal' END anomaly_level
FROM silver_country_streams_5m s JOIN regions r USING (country_code) JOIN h USING (ts, country_code) LEFT JOIN p USING (ts, country_code)""")
spark.sql("""
CREATE OR REPLACE TABLE gold_frustration_index_5m AS
WITH c AS (
  SELECT 'Country' scope_type, country_code scope, ts, sum(streams) streams, sum(baseline_streams) baseline_streams, sum(tickets) tickets, sum(tickets_15m) tickets_15m, sum(expected_tickets_15m) expected_tickets_15m FROM gold_region_heatmap_5m GROUP BY ALL
  UNION ALL SELECT 'Region', region_group, ts, sum(streams), sum(baseline_streams), sum(tickets), sum(tickets_15m), sum(expected_tickets_15m) FROM gold_region_heatmap_5m GROUP BY ALL
  UNION ALL SELECT 'Global', 'Global', ts, sum(streams), sum(baseline_streams), sum(tickets), sum(tickets_15m), sum(expected_tickets_15m) FROM gold_region_heatmap_5m GROUP BY ALL),
s AS (
  SELECT 'Country' scope_type, country_code scope, ts_15m, count(*) neg FROM gold_social_posts WHERE sentiment='negative' AND topic_domain<>'Show hype / fandom' AND country_code IS NOT NULL GROUP BY ALL
  UNION ALL SELECT 'Region', region_group, ts_15m, count(*) FROM gold_social_posts WHERE sentiment='negative' AND topic_domain<>'Show hype / fandom' GROUP BY ALL
  UNION ALL SELECT 'Global','Global', ts_15m, count(*) FROM gold_social_posts WHERE sentiment='negative' AND topic_domain<>'Show hype / fandom' GROUP BY ALL),
m AS (SELECT c.*, u.expected_release_uplift, u.active_release, coalesce(s.neg,0) negative_social_15m, c.streams/c.baseline_streams-1 stream_dev,
        c.tickets_15m/(c.streams/1e4)/3 contacts_per_10k_streams, c.expected_tickets_15m/(c.streams/1e4)/3 baseline_contacts_per_10k_streams,
        (c.tickets_15m+1)/(c.expected_tickets_15m+1) contact_ratio
      FROM c JOIN gold_release_uplift_5m u USING (ts) LEFT JOIN s ON s.scope_type=c.scope_type AND s.scope=c.scope
        AND s.ts_15m = CAST(from_unixtime(floor(unix_timestamp(c.ts)/900)*900) AS TIMESTAMP))
SELECT scope_type, scope, ts, streams, baseline_streams, round(stream_dev,4) stream_deviation_pct, expected_release_uplift, active_release,
  tickets, tickets_15m, round(expected_tickets_15m,1) expected_tickets_15m, round(contacts_per_10k_streams,3) contacts_per_10k_streams,
  round(baseline_contacts_per_10k_streams,3) baseline_contacts_per_10k_streams, round(contact_ratio,2) contact_ratio, negative_social_15m,
  round(least(100, 100*(0.55*least(1, greatest(0,contact_ratio-1)/4) + 0.25*least(1, negative_social_15m/10.0) + 0.20*least(1, greatest(0,-stream_dev)/0.25))),1) frustration_index,
  (stream_dev <= -0.08 AND contact_ratio >= 2) AS divergence_flag
FROM m""")

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE gold_signature_trends_15m AS
WITH t AS (SELECT CAST(from_unixtime(floor(unix_timestamp(created_at)/900)*900) AS TIMESTAMP) ts_15m, issue_signature, issue_domain, count(*) cnt FROM silver_tickets GROUP BY ALL),
hd AS (SELECT issue_signature, date(ts_15m) d, hour(ts_15m) h, sum(cnt) c FROM t GROUP BY ALL),
ht AS (SELECT issue_signature, h, sum(c) c FROM hd GROUP BY ALL),
b AS (SELECT t.*, greatest(0.3, (ht.c - coalesce(hd.c,0))/(14*4.0)) baseline_15m
      FROM t JOIN ht ON ht.issue_signature=t.issue_signature AND ht.h=hour(t.ts_15m)
      LEFT JOIN hd ON hd.issue_signature=t.issue_signature AND hd.d=date(t.ts_15m) AND hd.h=hour(t.ts_15m))
SELECT ts_15m, issue_signature, issue_domain, cnt AS tickets_15m, round(baseline_15m,2) baseline_15m, round((cnt-baseline_15m)/sqrt(baseline_15m),2) z_score, round(cnt/baseline_15m,1) surge_ratio,
  (cnt>=10 AND cnt>=4*baseline_15m AND (cnt-baseline_15m)/sqrt(baseline_15m)>=5) AS is_anomalous
FROM b""")
spark.sql("""
CREATE OR REPLACE TABLE silver_anomaly_events AS
WITH f AS (SELECT * FROM gold_signature_trends_15m WHERE is_anomalous),
g AS (SELECT *, CASE WHEN lag(ts_15m) OVER (PARTITION BY issue_signature ORDER BY ts_15m) >= ts_15m - INTERVAL 30 MINUTES THEN 0 ELSE 1 END new_evt FROM f),
e AS (SELECT *, sum(new_evt) OVER (PARTITION BY issue_signature ORDER BY ts_15m) evt_no FROM g)
SELECT concat('ANM-', date_format(min(ts_15m),'MMdd-HHmm'), '-', substr(sha1(issue_signature),1,4)) anomaly_id, issue_signature, issue_domain,
  min(ts_15m) first_seen, max(ts_15m) + INTERVAL 15 MINUTES last_seen, max(tickets_15m) peak_tickets_15m, max(surge_ratio) peak_surge_ratio, max(z_score) peak_z,
  round(avg(baseline_15m),2) baseline_15m, count(*) anomalous_windows
FROM e GROUP BY issue_signature, issue_domain, evt_no""")
spark.sql("""
CREATE OR REPLACE TABLE gold_ticket_anomalies_base AS
WITH tk AS (SELECT e.anomaly_id, t.* FROM silver_anomaly_events e JOIN silver_tickets t ON t.issue_signature=e.issue_signature AND t.created_at>=e.first_seen AND t.created_at<e.last_seen),
agg AS (SELECT anomaly_id, count(*) total_tickets, count(DISTINCT country_code) n_countries, array_join(collect_set(region_group),', ') region_groups,
          avg(CASE WHEN subscriber_tier IN ('Mega Fan','Ultimate Fan') THEN 1 ELSE 0 END) premium_share, avg(CAST(sla_met AS INT)) sla_met_rate, avg(csat_score) avg_csat,
          slice(collect_list(concat('[',country_code,'/',device,'/',language,'] ',subject,': ',description)),1,12) sample_tickets FROM tk GROUP BY anomaly_id),
cty AS (SELECT anomaly_id, array_join(slice(sort_array(collect_list(named_struct('k',-n,'c',concat(country_code,' (',n,')')))), 1, 5).c, ', ') top_countries
        FROM (SELECT anomaly_id, country_code, count(*) n FROM tk GROUP BY ALL) GROUP BY anomaly_id),
dev AS (SELECT anomaly_id, array_join(slice(sort_array(collect_list(named_struct('k',-n,'c',concat(device,' (',round(100*n/tot),'%)')))), 1, 3).c, ', ') top_devices
        FROM (SELECT anomaly_id, device, count(*) n, sum(count(*)) OVER (PARTITION BY anomaly_id) tot FROM tk GROUP BY ALL) GROUP BY anomaly_id),
inc AS (SELECT e.anomaly_id, max_by(i.incident_id, i.declared_at) incident_id, max(i.declared_at) incident_declared_at, max_by(i.title, i.declared_at) incident_title
        FROM silver_anomaly_events e JOIN incidents i ON i.domain=e.issue_domain AND i.declared_at BETWEEN e.first_seen - INTERVAL 60 MINUTES AND e.last_seen GROUP BY e.anomaly_id),
chg AS (SELECT e.anomaly_id, max_by(c.change_id, c.change_ts) change_id, max_by(concat(c.service,' ',c.version,' — ',c.description), c.change_ts) change_summary, max(c.change_ts) change_ts
        FROM silver_anomaly_events e JOIN change_events c ON c.domain IN (e.issue_domain, CASE WHEN e.issue_domain='Subtitles & Localization' THEN 'Content Ingestion' END)
         AND c.change_ts BETWEEN e.first_seen - INTERVAL 120 MINUTES AND e.first_seen + INTERVAL 15 MINUTES AND c.change_type <> 'rollback' AND c.outcome <> 'success' GROUP BY e.anomaly_id)
SELECT e.*, e.first_seen + INTERVAL 15 MINUTES AS detected_at, a.total_tickets, a.n_countries, cty.top_countries, dev.top_devices, a.region_groups,
  round(a.premium_share,3) premium_share, round(a.sla_met_rate,3) sla_met_rate, round(a.avg_csat,2) avg_csat, a.sample_tickets,
  inc.incident_id, inc.incident_title, inc.incident_declared_at,
  round((unix_timestamp(inc.incident_declared_at) - unix_timestamp(e.first_seen + INTERVAL 15 MINUTES))/60) AS minutes_detected_before_incident,
  chg.change_id AS suspected_change_id, chg.change_summary AS suspected_change, chg.change_ts AS suspected_change_ts
FROM silver_anomaly_events e JOIN agg a USING (anomaly_id) JOIN cty USING (anomaly_id) JOIN dev USING (anomaly_id) LEFT JOIN inc USING (anomaly_id) LEFT JOIN chg USING (anomaly_id)
WHERE e.issue_signature NOT LIKE '% · general'""")

In [0]:
%sql
CREATE OR REPLACE TABLE classic_stable_l0m1dn_catalog.support_command_center.gold_ticket_anomalies AS
WITH x AS (
SELECT a.*,
  ai_query('databricks-claude-sonnet-4-6',
    concat('You are an incident analyst for Fanimation, an anime streaming service. A surge of support tickets was detected. ',
      'Produce: headline (max 12 words), summary (2-3 concise sentences, plain English, mention scale, geography, device and likely cause), ',
      'likely_root_cause (1 sentence), recommended_actions (3 short imperative strings for engineering + support), customer_macro (1-2 sentence known-issue reply agents can send to customers, in English).',
      '\nSignature: ', issue_signature, '\nDomain: ', issue_domain, '\nDetected at: ', cast(detected_at as string), ' UTC; active until ', cast(last_seen as string),
      '\nTotal tickets: ', total_tickets, '; peak per 15 min: ', peak_tickets_15m, ' vs baseline ', baseline_15m,
      '\nTop countries: ', top_countries, '\nTop devices: ', top_devices, '\nRecent change near onset: ', coalesce(suspected_change,'none found'),
      '\nSample tickets (multilingual): ', array_join(sample_tickets, ' || ')),
    responseFormat => 'STRUCT<analysis:STRUCT<headline:STRING, summary:STRING, likely_root_cause:STRING, recommended_actions:ARRAY<STRING>, customer_macro:STRING>>') AS raw
FROM classic_stable_l0m1dn_catalog.support_command_center.gold_ticket_anomalies_base a),
p AS (SELECT * EXCEPT (raw), from_json(raw, 'headline STRING, summary STRING, likely_root_cause STRING, recommended_actions ARRAY<STRING>, customer_macro STRING') j FROM x)
SELECT * EXCEPT (j, sample_tickets), j.headline ai_headline, j.summary ai_summary, j.likely_root_cause ai_likely_root_cause,
  j.recommended_actions ai_recommended_actions, j.customer_macro ai_customer_macro, sample_tickets
FROM p

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE gold_incident_scorecard AS
WITH i AS (SELECT * FROM incidents),
cs AS (SELECT i.incident_id, min(h.ts) first_customer_signal_at FROM i JOIN gold_service_health_country_5m h ON h.domain=i.domain AND array_contains(split(i.countries,','), h.country_code)
        AND h.ts BETWEEN i.declared_at - INTERVAL 3 HOURS AND i.resolved_at AND h.customer_signal GROUP BY i.incident_id),
an AS (SELECT incident_id, min(detected_at) first_ticket_anomaly_at, max_by(anomaly_id, total_tickets) anomaly_id, max_by(issue_signature, total_tickets) issue_signature,
          max_by(ai_headline, total_tickets) ai_headline FROM gold_ticket_anomalies GROUP BY incident_id),
tk AS (SELECT i.incident_id, count(*) tickets_during_incident, avg(t.csat_score) csat_during_incident, avg(CAST(t.sla_met AS INT)) sla_met_rate
       FROM i JOIN silver_tickets t ON t.issue_domain=i.domain AND array_contains(split(i.countries,','), t.country_code)
        AND t.created_at BETWEEN coalesce(i.first_alert_at, i.declared_at) - INTERVAL 90 MINUTES AND i.resolved_at GROUP BY i.incident_id),
st AS (SELECT i.incident_id, sum(CASE WHEN s.streams < 0.95*s.baseline_streams THEN s.baseline_streams - s.streams ELSE 0 END) lost_stream_intervals
       FROM i JOIN silver_country_streams_5m s ON array_contains(split(i.countries,','), s.country_code) AND s.ts BETWEEN i.declared_at - INTERVAL 90 MINUTES AND i.resolved_at GROUP BY i.incident_id),
pay AS (SELECT i.incident_id, sum(p.revenue_at_risk_usd) revenue_at_risk_usd FROM i JOIN gold_payment_health_5m p ON i.domain='Billing & Payments'
        AND array_contains(split(i.countries,','), p.country_code) AND p.ts BETWEEN i.declared_at - INTERVAL 90 MINUTES AND i.resolved_at GROUP BY i.incident_id),
so AS (SELECT i.incident_id, count(*) negative_social_posts FROM i JOIN gold_social_posts g ON g.sentiment='negative'
        AND g.posted_at BETWEEN i.declared_at - INTERVAL 90 MINUTES AND i.resolved_at AND array_contains(split(i.countries,','), g.country_code) GROUP BY i.incident_id)
SELECT i.*, cs.first_customer_signal_at, an.first_ticket_anomaly_at, an.anomaly_id, an.issue_signature, an.ai_headline,
  least(cs.first_customer_signal_at, an.first_ticket_anomaly_at) customer_detected_at,
  round((unix_timestamp(coalesce(i.first_alert_at, i.declared_at)) - unix_timestamp(least(cs.first_customer_signal_at, an.first_ticket_anomaly_at)))/60) detection_lead_minutes,
  CASE WHEN least(cs.first_customer_signal_at, an.first_ticket_anomaly_at) < coalesce(i.first_alert_at, i.declared_at) THEN 'Customers first' ELSE 'Monitoring first' END detection_winner,
  round((unix_timestamp(i.resolved_at)-unix_timestamp(i.declared_at))/60) minutes_to_resolve,
  coalesce(tk.tickets_during_incident,0) tickets_during_incident, round(tk.csat_during_incident,2) csat_during_incident, round(tk.sla_met_rate,3) sla_met_rate,
  round(st.lost_stream_intervals*5/60) est_lost_viewing_hours, round(coalesce(pay.revenue_at_risk_usd,0),2) revenue_at_risk_usd, coalesce(so.negative_social_posts,0) negative_social_posts
FROM i LEFT JOIN cs USING (incident_id) LEFT JOIN an USING (incident_id) LEFT JOIN tk USING (incident_id) LEFT JOIN st USING (incident_id) LEFT JOIN pay USING (incident_id) LEFT JOIN so USING (incident_id)""")
spark.sql("""
CREATE OR REPLACE TABLE gold_exec_daily_kpis AS
WITH t AS (SELECT date(created_at) d, region_group, count(*) tickets, avg(first_response_minutes) avg_first_response_min, percentile_approx(first_response_minutes,0.9) p90_first_response_min,
            avg(CAST(sla_met AS INT)) sla_met_rate, avg(csat_score) avg_csat, avg(CAST(reopened AS INT)) reopen_rate,
            avg(CASE WHEN subscriber_tier IN ('Mega Fan','Ultimate Fan') THEN 1 ELSE 0 END) premium_ticket_share FROM silver_tickets GROUP BY ALL),
s AS (SELECT date(s.ts) d, r.region_group, sum(streams) stream_intervals FROM silver_country_streams_5m s JOIN regions r USING (country_code) GROUP BY ALL),
sp AS (SELECT date(ts) d, scope region_group, max(streams) peak_concurrent_streams, avg(frustration_index) avg_frustration_index, max(frustration_index) max_frustration_index
       FROM gold_frustration_index_5m WHERE scope_type='Region' GROUP BY ALL),
h AS (SELECT date(ts) d, region_group, 5*sum(CASE WHEN health_state='Customer-Detected' THEN 1 ELSE 0 END) customer_detected_minutes,
            5*sum(CASE WHEN health_state='Known Incident' THEN 1 ELSE 0 END) known_incident_minutes FROM gold_service_health_region_5m GROUP BY ALL),
p AS (SELECT date(ts) d, region_group, sum(successes)/sum(attempts) payment_success_rate, sum(revenue_at_risk_usd) revenue_at_risk_usd FROM gold_payment_health_5m GROUP BY ALL),
so AS (SELECT date(posted_at) d, region_group, count(*) social_posts, avg(CASE WHEN sentiment='negative' THEN 1 ELSE 0 END) negative_social_share FROM gold_social_posts WHERE region_group IS NOT NULL GROUP BY ALL),
inc AS (SELECT date(declared_at) d, rg region_group, count(*) incidents_declared FROM incidents LATERAL VIEW explode(split(region_groups,',')) x AS rg GROUP BY ALL)
SELECT t.d AS kpi_date, t.region_group, t.tickets, round(t.tickets/(s.stream_intervals/1e4),3) contacts_per_10k_stream_intervals,
  round(t.avg_first_response_min,1) avg_first_response_min, round(t.p90_first_response_min,1) p90_first_response_min, round(t.sla_met_rate,3) sla_met_rate,
  round(t.avg_csat,2) avg_csat, round(t.reopen_rate,3) reopen_rate, round(t.premium_ticket_share,3) premium_ticket_share,
  sp.peak_concurrent_streams, round(sp.avg_frustration_index,1) avg_frustration_index, sp.max_frustration_index,
  coalesce(h.customer_detected_minutes,0) customer_detected_minutes, coalesce(h.known_incident_minutes,0) known_incident_minutes,
  round(p.payment_success_rate,4) payment_success_rate, round(p.revenue_at_risk_usd,2) revenue_at_risk_usd,
  coalesce(so.social_posts,0) social_posts, round(so.negative_social_share,3) negative_social_share, coalesce(inc.incidents_declared,0) incidents_declared
FROM t JOIN s USING (d, region_group) JOIN sp USING (d, region_group) LEFT JOIN h USING (d, region_group) LEFT JOIN p USING (d, region_group)
LEFT JOIN so USING (d, region_group) LEFT JOIN inc USING (d, region_group)""")